## Trening na Colabie → automatycznie na PC (przez Google Drive)

Colab w VS Code **nie umie** zapisać plików prosto na dysk lokalny z komórki Pythona (`files.download` działa tylko w przeglądarce).

Działający automatyczny most:
1. Trening leci na GPU Colaba.
2. Wyniki zapisują się na **Google Drive**.
3. **Google Drive na komputer** synchronizuje je na Twój dysk (np. `G:\\My Drive\\Battleships`).

### Jednorazowa konfiguracja
1. Zainstaluj [Google Drive na komputer](https://www.google.com/drive/download/) i zaloguj się tym samym kontem co Colab.
2. W VS Code: wtyczka **Colab** → otwórz ten notebook → **Select Kernel** → **Colab** → **Auto Connect**.
3. Odpal komórki poniżej po kolei.

## 1. Montowanie Google Drive

Albo odpal komórkę poniżej, albo: `Ctrl+Shift+P` → **Colab: Mount Google Drive to Server...** (dopisze podobny snippet).

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 2. Kod na serwerze Colaba + folder wyników na Drive

Kod klonujemy na szybki dysk Colaba (`/content`). Wyniki (`models`, `plots`) idą na Drive, żeby Drive Desktop zaciągnął je na PC.

In [ ]:
import os
import shutil
from pathlib import Path

REPO_URL = "https://github.com/Slasq/Battleships.git"
BRANCH = "RL-and-DQN"

# Ścieżka na Colabie (trening)
PROJECT = Path("/content/Battleships")

# Ta sama struktura na Drive → synchronizuje się na PC przez Drive Desktop
DRIVE_ROOT = Path("/content/drive/MyDrive/Battleships")
DRIVE_MODELS = DRIVE_ROOT / "models"
DRIVE_PLOTS = DRIVE_ROOT / "plots"

DRIVE_MODELS.mkdir(parents=True, exist_ok=True)
DRIVE_PLOTS.mkdir(parents=True, exist_ok=True)

os.chdir("/content")
if PROJECT.is_dir() and (PROJECT / "ml").is_dir():
    print(f"Projekt już jest: {PROJECT}")
else:
    !git clone --branch {BRANCH} --single-branch {REPO_URL} Battleships

assert (PROJECT / "ml" / "dqn" / "train.py").exists(), (
    "Brak projektu. Sprawdź clone albo Upload to Colab."
)

os.chdir(PROJECT)
print("CWD:", os.getcwd())
print("Wyniki → Drive:", DRIVE_ROOT)
print("Na PC szukaj w folderze Google Drive: My Drive/Battleships/")

In [ ]:
def sync_artifacts_to_drive():
    """Kopiuje models/ i plots/ z Colaba na Drive (stamtąd lecą na PC)."""
    local_models = PROJECT / "models"
    local_plots = PROJECT / "plots"

    if local_models.exists():
        shutil.copytree(local_models, DRIVE_MODELS, dirs_exist_ok=True)
        print("Zsynchronizowano models →", DRIVE_MODELS)
    else:
        print("Brak folderu models — pomijam")

    if local_plots.exists():
        shutil.copytree(local_plots, DRIVE_PLOTS, dirs_exist_ok=True)
        print("Zsynchronizowano plots →", DRIVE_PLOTS)
    else:
        print("Brak folderu plots — pomijam")

    print("Gotowe. Drive Desktop zaraz zaciągnie pliki na komputer.")

print("sync_artifacts_to_drive() gotowe do użycia po treningu")

## 3. Trening DQN

Po zakończeniu komórka sama wrzuca wyniki na Drive → PC.

In [ ]:
os.chdir(PROJECT)
!python ml/dqn/train.py
sync_artifacts_to_drive()

## 4. Trening Probmap

In [ ]:
os.chdir(PROJECT)
!python ml/probmap/train.py
sync_artifacts_to_drive()

## 5. Ewaluacja (opcjonalnie)

In [ ]:
os.chdir(PROJECT)
!python ml/evaluate.py
sync_artifacts_to_drive()

## Gdzie są pliki na PC?

Po syncu Drive Desktop:
- zwykle `G:\\My Drive\\Battleships\\models` i `...\\plots`, albo
- `C:\\Users\\<Ty>\\Google Drive\\My Drive\\Battleships\\...`

Możesz skopiować je do lokalnego repo `Desktop\\Battleships`, albo trzymać projekt na Drive i otwierać ten folder w VS Code.

### Alternatywa bez Drive (ręcznie)
Settings → włącz **Colab › Experimental: Server Mounting** → `Colab: Mount Server To Workspace` → po treningu w zamontowanym `/content` PPM na `models` → **Download**.